In [179]:
import pandas as pd

In [180]:
# загружаем датасет
data = pd.read_csv("./data/task_2_data_ex.csv")
data.head()

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10


In [181]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 11 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                1320 non-null   int64  
 1   month                               1320 non-null   int64  
 2   produced_material                   1320 non-null   int64  
 3   produced_material_production_type   1320 non-null   int64  
 4   produced_material_release_type      1320 non-null   object 
 5   produced_material_quantity          1320 non-null   object 
 6   component_material                  1320 non-null   int64  
 7   component_material_production_type  480 non-null    float64
 8   component_material_release_type     1320 non-null   object 
 9   component_material_quantity         1320 non-null   object 
 10  plant_id                            1320 non-null   object 
dtypes: float64(1), int64(5), object(5)
memory u

In [182]:
data.loc[data['produced_material_release_type'] == 'FIN', 'produced_material'].unique()

array([10000, 10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008,
       10009])

In [183]:
def parse_financial_num(col):
    if col.dtype == 'object':
        # 1. Убираем запятые (разделители тысяч)
        # 2. Убираем лишние пробелы по краям
        return col.str.replace(',', '', regex=False).str.strip()
    return col

data['produced_qty_clean'] = pd.to_numeric(parse_financial_num(data['produced_material_quantity']), errors='coerce')
data['component_qty_clean'] = pd.to_numeric(parse_financial_num(data['component_material_quantity']), errors='coerce')

data['produced_material_quantity'] = data['produced_qty_clean'].fillna(0)
data['component_material_quantity'] = data['component_qty_clean'].fillna(0)

data = data.groupby([
    'plant_id', 'year', 
    'produced_material', 'produced_material_release_type', 'produced_material_production_type',
    'component_material', 'component_material_release_type', 'component_material_production_type'
], observed=True).agg({
    'produced_material_quantity': 'sum',
    'component_material_quantity': 'sum'
}).reset_index()

print(f"Записей после годовой агрегации: {len(data)}")

Записей после годовой агрегации: 40


In [184]:
data

,plant_id,year,produced_material,produced_material_release_type,produced_material_production_type,component_material,component_material_release_type,component_material_production_type,produced_material_quantity,component_material_quantity
0,RLT_10,2024,10000,FIN,8002,50000,PROD,8002.0,11708.0,11708.0
1,RLT_10,2024,10001,FIN,8002,50001,PROD,8002.0,12023.0,12023.0
2,RLT_10,2024,50000,PROD,8002,80070,PROD,8007.0,9538.0,11303.0
3,RLT_10,2024,50001,PROD,8002,80071,PROD,8007.0,9487.0,10759.0
4,RLT_10,2024,80010,PROD,8001,80000,PROD,8000.0,21013.0,23360.0
5,RLT_10,2024,80011,PROD,8001,80001,PROD,8000.0,21300.0,24730.0
6,RLT_10,2024,80070,PROD,8007,80010,PROD,8001.0,11028.0,41769.0
7,RLT_10,2024,80071,PROD,8007,80011,PROD,8001.0,10751.0,42650.0
8,RLT_14,2024,10002,FIN,8002,50002,PROD,8002.0,12067.0,12067.0
9,RLT_14,2024,10003,FIN,8002,50003,PROD,8002.0,12091.0,12091.0


In [185]:

cols_to_use = [
    'plant_id', 'year', 
    'produced_material', 'produced_material_release_type', 
    'produced_material_production_type', 'produced_material_quantity',
    'component_material', 'component_material_release_type', 
    'component_material_production_type', 'component_material_quantity'
]

df = data[cols_to_use].copy()

first_level = df[df['produced_material_release_type'] == 'FIN'].copy()


first_level = first_level.rename(columns={
    'produced_material': 'fin_material_id',
    'produced_material_release_type': 'fin_material_release_type',
    'produced_material_production_type': 'fin_material_production_type',
    'produced_material_quantity': 'fin_production_quantity',
    
    'component_material': 'prod_material_id',
    'component_material_release_type': 'prod_material_release_type',
    'component_material_production_type': 'prod_material_production_type',
    'component_material_quantity': 'prod_production_quantity'
})

first_level = first_level.merge(
    df, 
    left_on=['plant_id', 'year', 'prod_material_id'], 
    right_on=['plant_id', 'year', 'produced_material'], 
    how='left', 
    suffixes=('', '_component')
)

first_level = first_level.rename(columns={
    'component_material': 'component_id',
    'component_material_release_type': 'component_material_release_type',
    'component_material_production_type': 'component_material_production_type',
    'component_material_quantity': 'component_consumption_quantity'
})

existing_cols = [
    'plant_id', 
    'year', 
    'fin_material_id', 
    'fin_material_release_type', 
    'fin_material_production_type', 
    'fin_production_quantity',
    'prod_material_id', 
    'prod_material_release_type', 
    'prod_material_production_type', 
    'prod_production_quantity',
    'component_id',                   
    'component_material_release_type', 
    'component_material_production_type', 
    'component_consumption_quantity'
]

first_level = first_level[existing_cols]

print(first_level.info())
first_level.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   plant_id                            10 non-null     object 
 1   year                                10 non-null     int64  
 2   fin_material_id                     10 non-null     int64  
 3   fin_material_release_type           10 non-null     object 
 4   fin_material_production_type        10 non-null     int64  
 5   fin_production_quantity             10 non-null     float64
 6   prod_material_id                    10 non-null     int64  
 7   prod_material_release_type          10 non-null     object 
 8   prod_material_production_type       10 non-null     float64
 9   prod_production_quantity            10 non-null     float64
 10  component_id                        10 non-null     int64  
 11  component_material_release_type     10 non-null 

,plant_id,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity
0,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,80070,PROD,8007.0,11303.0
1,RLT_10,2024,10001,FIN,8002,12023.0,50001,PROD,8002.0,12023.0,80071,PROD,8007.0,10759.0
2,RLT_14,2024,10002,FIN,8002,12067.0,50002,PROD,8002.0,12067.0,80072,PROD,8007.0,11027.0
3,RLT_14,2024,10003,FIN,8002,12091.0,50003,PROD,8002.0,12091.0,80073,PROD,8007.0,10806.0
4,RLT_14,2024,10004,FIN,8002,12091.0,50004,PROD,8002.0,12091.0,80074,PROD,8007.0,10806.0


In [ ]:
df_next = df.rename(columns={
    'produced_material': 'component_id',  # ключ для связи (будет совпадать с current_level.component_id)
    'produced_material_release_type': 'pm_rt_next',
    'produced_material_production_type': 'pm_pt_next',
    'produced_material_quantity': 'pm_qty_next',
    'component_material': 'cm_id_next',
    'component_material_release_type': 'cm_rt_next',
    'component_material_production_type': 'cm_pt_next',
    'component_material_quantity': 'cm_qty_next',
})

current_level = first_level[first_level['component_material_release_type'] == 'PROD'].copy()

current_level['level'] = 0
level_counter = 0

all_prod_levels = []

while not current_level.empty:
    current_level['level'] = level_counter

    cols_to_save = [c for c in existing_cols + ['level'] if c in current_level.columns]
    if not cols_to_save:
        break
    all_prod_levels.append(current_level[cols_to_save].copy())

    if 'component_id' in current_level.columns:
        left_on = ['plant_id', 'year', 'component_id']
        right_on = ['plant_id', 'year', 'component_id']
    elif 'component_material' in current_level.columns:
        left_on = ['plant_id', 'year', 'component_material']
        right_on = ['plant_id', 'year', 'component_id']
    else:
        break

    next_step = current_level.merge(
        df_next,
        left_on=left_on,
        right_on=right_on,
        how='inner' 
    )

    if next_step.empty:
        break

    next_step = next_step.assign(
        prod_material_id = next_step.get('component_id'),
        prod_material_release_type = next_step.get('component_material_release_type'),
        prod_material_production_type = next_step.get('component_material_production_type'),
        prod_production_quantity = next_step.get('component_consumption_quantity'),
        
        component_id = next_step.get('cm_id_next'),
        component_material_release_type = next_step.get('cm_rt_next'),
        component_material_production_type = next_step.get('cm_pt_next'),
        component_consumption_quantity = next_step.get('cm_qty_next'),
    )

    next_step['level'] = level_counter + 1
    
    current_level = next_step[[c for c in existing_cols + ['level'] if c in next_step.columns]].copy()
    
    if 'component_material_release_type' in current_level.columns and (current_level['component_material_release_type'] != 'PROD').all():
        all_prod_levels.append(current_level)
        break

    level_counter += 1

if all_prod_levels:
    first_level_expanded = pd.concat(all_prod_levels, ignore_index=True).drop_duplicates()
else:
    first_level_expanded = pd.DataFrame(columns=existing_cols + ['level'])

print(f"Итого строк в иерархии: {len(first_level_expanded)}")

Итого строк в иерархии: 30


In [187]:
first_level_expanded.head()

,plant_id,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,level
0,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,80070,PROD,8007.0,11303.0,0
1,RLT_10,2024,10001,FIN,8002,12023.0,50001,PROD,8002.0,12023.0,80071,PROD,8007.0,10759.0,0
2,RLT_14,2024,10002,FIN,8002,12067.0,50002,PROD,8002.0,12067.0,80072,PROD,8007.0,11027.0,0
3,RLT_14,2024,10003,FIN,8002,12091.0,50003,PROD,8002.0,12091.0,80073,PROD,8007.0,10806.0,0
4,RLT_14,2024,10004,FIN,8002,12091.0,50004,PROD,8002.0,12091.0,80074,PROD,8007.0,10806.0,0


In [188]:
first_level_expanded.loc[first_level_expanded['fin_material_id'] == 10000, 'component_id'].unique()

array([80070, 80010, 80000])

In [189]:
first_level_expanded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 15 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   plant_id                            30 non-null     object 
 1   year                                30 non-null     int64  
 2   fin_material_id                     30 non-null     int64  
 3   fin_material_release_type           30 non-null     object 
 4   fin_material_production_type        30 non-null     int64  
 5   fin_production_quantity             30 non-null     float64
 6   prod_material_id                    30 non-null     int64  
 7   prod_material_release_type          30 non-null     object 
 8   prod_material_production_type       30 non-null     float64
 9   prod_production_quantity            30 non-null     float64
 10  component_id                        30 non-null     int64  
 11  component_material_release_type     30 non-null

In [190]:

sort_keys = [c for c in ['plant_id','year','fin_material_id','level','prod_material_id','component_id'] if c in first_level_expanded.columns]
if sort_keys:
    first_level_expanded = first_level_expanded.sort_values(sort_keys).reset_index(drop=True)
else:
    print('No sort keys found; showing unsorted result')

first_level_expanded.head(3)

,plant_id,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,level
0,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,80070,PROD,8007.0,11303.0,0
1,RLT_10,2024,10000,FIN,8002,11708.0,80070,PROD,8007.0,11303.0,80010,PROD,8001.0,41769.0,1
2,RLT_10,2024,10000,FIN,8002,11708.0,80010,PROD,8001.0,41769.0,80000,PROD,8000.0,23360.0,2


In [ ]:

column_mapping = {
    'plant_id': 'plant',
    'fin_material_id': 'fin_material_id',
    'fin_material_release_type': 'fin_material_release_type',
    'fin_material_production_type': 'fin_material_production_type',
    'fin_production_quantity': 'fin_production_quantity',
    'prod_material_id': 'prod_material_id',
    'prod_material_release_type': 'prod_material_release_type',
    'prod_material_production_type': 'prod_material_production_type',
    'prod_production_quantity': 'prod_material_production_quantity',
    'component_id': 'component_id',
    'component_material_release_type': 'component_material_release_type',
    'component_material_production_type': 'component_material_production_type',
    'component_consumption_quantity': 'component_consumption_quantity',
    'year': 'year'
}

out = first_level_expanded.copy()
out = out.rename(columns=column_mapping)

final_columns = [
    'plant',
    'fin_material_id',
    'fin_material_release_type',
    'fin_material_production_type',
    'fin_production_quantity',
    'prod_material_id',
    'prod_material_release_type',
    'prod_material_production_type',
    'prod_material_production_quantity',
    'component_id',
    'component_material_release_type',
    'component_material_production_type',
    'component_consumption_quantity',
    'year'
]


out = out[final_columns]

out_path = './data/bom_exploded_result_pandas.csv'
out.to_csv(out_path, index=False)

print(f"Результат сохранен в {out_path}")
print(f"Строк: {len(out)}")
print(f"Колонки: {list(out.columns)}")

Результат сохранен в ./data/bom_exploded_result.csv
Строк: 30
Колонки: ['plant', 'fin_material_id', 'fin_material_release_type', 'fin_material_production_type', 'fin_production_quantity', 'prod_material_id', 'prod_material_release_type', 'prod_material_production_type', 'prod_material_production_quantity', 'component_id', 'component_material_release_type', 'component_material_production_type', 'component_consumption_quantity', 'year']
